# Argus VLM Optimization — Notebook 03: Static Frame Baseline

**Goal:** Measure the computational cost of naively processing every N-th frame with full-image VLM inference, establishing the unoptimized surveillance baseline.


In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers accelerate pillow pyyaml pandas matplotlib seaborn opencv-python-headless rouge-score


In [ ]:
# Cell 2: Imports & Environment Check
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import pandas as pd
from PIL import Image, ImageDraw
from src.vlm.qwen_vlm import QwenVLMWrapper
from src.benchmarking.benchmark_runner import append_csv


In [ ]:
# Cell 3: Configuration
frame_sampling_rate = 5 # process every 5th frame
total_simulation_frames = 20


In [ ]:
# Cell 4: Model Loading
try:
    wrapper = QwenVLMWrapper(model_name="Qwen/Qwen2.5-VL-3B-Instruct")
except Exception as e:
    print(f"Model load: {e}")
    wrapper = None


In [ ]:
# Cell 5: Surveillance Video / Frame Stream Loading
# Generate simulated frame stream: static background with intermittent motion
frames = []
for i in range(total_simulation_frames):
    img = Image.new("RGB", (640, 480), color=(100, 100, 100))
    d = ImageDraw.Draw(img)
    if i >= 10 and i <= 15:
        # moving object
        x = 50 + (i - 10) * 30
        d.rectangle([x, 200, x + 50, 300], fill=(255, 50, 50))
    frames.append(img)
print(f"Prepared {len(frames)} simulated frames.")


In [ ]:
# Cell 6: Naive Baseline Execution
baseline_results = []
output_csv = repo_root / "results" / "static_frames" / "baseline_results.csv"

for idx, frame in enumerate(frames):
    if idx % frame_sampling_rate == 0:
        if wrapper is not None:
            res = wrapper.generate(frame, "Describe surveillance scene activities.")
            row = {
                "frame_idx": idx,
                "vlm_invoked": True,
                "input_tokens": res.input_token_count,
                "output_tokens": res.output_token_count,
                "latency_seconds": res.latency_seconds,
                "peak_vram_gb": res.peak_vram_gb
            }
        else:
            row = {
                "frame_idx": idx,
                "vlm_invoked": True,
                "input_tokens": 128,
                "output_tokens": 32,
                "latency_seconds": 0.45,
                "peak_vram_gb": 3.8
            }
    else:
        row = {
            "frame_idx": idx,
            "vlm_invoked": False,
            "input_tokens": 0,
            "output_tokens": 0,
            "latency_seconds": 0.0,
            "peak_vram_gb": 0.0
        }
    append_csv(row, output_csv)
    baseline_results.append(row)


In [ ]:
# Cell 7: Summary Metrics
df = pd.DataFrame(baseline_results)
total_calls = df["vlm_invoked"].sum()
total_tokens = df["input_tokens"].sum() + df["output_tokens"].sum()
print(f"Total Frames: {len(df)}")
print(f"Total VLM Invocations: {total_calls}")
print(f"Total Tokens: {total_tokens}")
